# Input × GR baseline (Output-Transformer dataset split)

This notebook evaluates the exported gain-reduction curve (`gr_curves/<setting>/<song>.pt`, key `gr_db`) as a reconstruction front-end:

`pred_wet = dry * 10**(gr_db_clamped/20)`

and compares `pred_wet` against the ground-truth wet audio using the same duration-weighted metrics/aggregation logic as `05_conditioning/eval_lstm_tfilm_gr.ipynb`.


In [1]:
import os
import sys
import json
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=UserWarning)

# Resolve repo root robustly (not dependent on where the notebook is launched from)
REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)

# Make sure `06_output/dataset.py` & friends shadow `05_conditioning/dataset.py`
sys.path.insert(0, str(REPO_ROOT / "06_output"))
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred"))

try:
    from dataset import discover_output_transformer_pairs
except ModuleNotFoundError as e:
    # `06_output/dataset.py` imports Lightning for the DataModule, but we only
    # need the lightweight filesystem discovery function here.
    if "lightning" not in str(e).lower():
        raise
    import glob

    def discover_output_transformer_pairs(data_root: str) -> list[dict]:
        """Fallback discovery (no Lightning import).

        Matches `06_output/dataset.py` logic: pairs have BOTH a GR curve
        and a wet WAV.
        """
        dry_lookup = {
            os.path.basename(p).replace("_UnmasteredWAV.wav", ""): p
            for p in glob.glob(
                os.path.join(data_root, "processed_normalized", "*_UnmasteredWAV.wav")
            )
        }
        pairs: list[dict] = []
        gr_root = os.path.join(data_root, "gr_curves")
        for setting in sorted(os.listdir(gr_root)):
            if not setting.startswith("threshold_"):
                continue
            wet_dir = os.path.join(data_root, "processed_ground_truth", setting)
            for pt in sorted(glob.glob(os.path.join(gr_root, setting, "*.pt"))):
                song = os.path.splitext(os.path.basename(pt))[0]
                wet = os.path.join(wet_dir, f"{song}-exported.wav")
                if song in dry_lookup and os.path.isfile(wet):
                    pairs.append(
                        {
                            "song": song,
                            "setting": setting,
                            "dry": dry_lookup[song],
                            "gr": pt,
                            "wet": wet,
                        }
                    )
        return sorted(pairs, key=lambda p: (p["song"], p["setting"]))

    print("[warn] Using fallback discover_output_transformer_pairs(): missing Lightning")

from splits import build_split_manifest
from amplitude_match import GR_DB_MIN, GR_DB_MAX

from eval_helpers import (
    _read_dry_wet_segment,
    _weighted_average_metric_rows,
    _pair_num_frames,
    _db_to_amplitude,
)

from src.dsp_torch import gain_reduction_db, RMS_WINDOW
from src.losses import compute_all_losses as compute_audio_losses

print(f"repo: {REPO_ROOT}")
print(f"torch: {torch.__version__}")


/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo: /Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling
torch: 2.10.0


In [2]:
# Diff-SSL-G-Comp
DATA_ROOT = "/Volumes/Saola's Drive/AllCode/thesis/data/Diff-SSL-G-Comp"
SAMPLE_RATE = 44100

# Matches the chunk alignment used in `05_conditioning/eval_lstm_tfilm_gr.ipynb`
# (loaded TFiLM GR model had hop_size=256, tfilm_block_size=8 => step=2048)
STREAM_CHUNK_SEC = 10.0
STEP = 256 * 8

# Set to a small integer for smoke tests
MAX_EVAL_PAIRS = None

metric_cols = [
    "GR MAE (dB)",
    "MAE (L1)",
    "MSE (L2)",
    "ESR",
    "MR-STFT",
    "EDC",
    "M_NRMSE",
    "M_SF",
]

assert os.path.isdir(DATA_ROOT), f"Missing DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), "Missing gr_curves folder"

pairs = discover_output_transformer_pairs(DATA_ROOT)
print(f"Discovered (song, setting) pairs: {len(pairs)}")

# Identical split logic to `06_output/train_lstm_output_transformer.ipynb`
SPLIT_SEED = 42
N_VAL_SONGS = 1
N_TEST_SONGS = 2
split = build_split_manifest(
    pairs,
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
)

VAL_PAIRS = [tuple(k.split("::", 1)) for k in split.val_pair_keys]
TEST_PAIRS = [tuple(k.split("::", 1)) for k in split.test_pair_keys]

print(f"Val pairs  : {len(VAL_PAIRS)} (songs={split.val_songs})")
print(f"Test pairs : {len(TEST_PAIRS)} (songs={split.test_songs})")

chunk_frames = int(round(STREAM_CHUNK_SEC * SAMPLE_RATE))
chunk_frames -= chunk_frames % STEP
print(f"Chunking: {STREAM_CHUNK_SEC}s ~ {chunk_frames} frames, step={STEP}")


Discovered (song, setting) pairs: 100
Val pairs  : 10 (songs=['Ecstasy'])
Test pairs : 4 (songs=['Air', 'AncoraQui'])
Chunking: 10.0s ~ 440320 frames, step=2048


In [3]:
def pair_paths(song: str, setting: str) -> tuple[str, str, str]:
    dry_path = os.path.join(DATA_ROOT, "processed_normalized", f"{song}_UnmasteredWAV.wav")
    wet_path = os.path.join(DATA_ROOT, "processed_ground_truth", setting, f"{song}-exported.wav")
    gr_path = os.path.join(DATA_ROOT, "gr_curves", setting, f"{song}.pt")
    return dry_path, wet_path, gr_path


@torch.no_grad()
def stream_pair_metrics(song: str, setting: str) -> list[dict]:
    """Compute duration-weighted metrics for one (song, setting) pair.

    Predicted wet is reconstructed *only* from the exported GR curve:
        pred_wet = dry * 10**(gr_db_clamped/20)
    """
    dry_path, wet_path, gr_path = pair_paths(song, setting)

    # One-time load per pair; GR curves live in RAM/disk as a per-sample tensor
    gr_obj = torch.load(gr_path, weights_only=False, map_location="cpu")
    gr_db_full = gr_obj["gr_db"].float()
    if gr_db_full.ndim == 1:
        gr_db_full = gr_db_full.unsqueeze(0)
    if gr_db_full.shape[0] > 1:
        gr_db_full = gr_db_full.mean(dim=0, keepdim=True)

    total = _pair_num_frames(dry_path, wet_path, SAMPLE_RATE)
    total = min(total, gr_db_full.shape[-1])

    rows: list[dict] = []
    for ci, o in enumerate(range(0, total, chunk_frames), start=1):
        dry, wet = _read_dry_wet_segment(
            dry_path, wet_path, o, min(o + chunk_frames, total), SAMPLE_RATE
        )

        # Mirror the eval notebook: crop segment length to a multiple of STEP
        L = dry.shape[-1] - (dry.shape[-1] % STEP)
        if L < STEP:
            break

        dry = dry[..., :L]
        wet = wet[..., :L]
        gr_db = gr_db_full[..., o : o + L].clamp(GR_DB_MIN, GR_DB_MAX)

        pred_wet = dry * _db_to_amplitude(gr_db)

        gr_target_db = gain_reduction_db(
            dry.unsqueeze(0), wet.unsqueeze(0), RMS_WINDOW
        ).squeeze(0).clamp(GR_DB_MIN, GR_DB_MAX)

        losses = compute_audio_losses(pred_wet.unsqueeze(0), wet.unsqueeze(0))

        rows.append(
            {
                "Chunk": ci,
                "Start (s)": o / SAMPLE_RATE,
                "Duration (s)": L / SAMPLE_RATE,
                "Frames": L,
                "GR MAE (dB)": float((gr_db - gr_target_db).abs().mean()),
                **losses,
            }
        )

    return rows


In [4]:
# ── Run evaluation ──────────────────────────────────────────────────────

pair_rows_all = []
split_rows_all = []

for split_name, split_pairs in (("validation", VAL_PAIRS), ("test", TEST_PAIRS)):
    pairs_to_eval = split_pairs
    if MAX_EVAL_PAIRS is not None:
        pairs_to_eval = pairs_to_eval[:MAX_EVAL_PAIRS]

    chunk_rows: list[dict] = []
    pair_rows: list[dict] = []

    for i, (song, setting) in enumerate(pairs_to_eval, start=1):
        print(f"[{split_name} {i}/{len(pairs_to_eval)}] {song} / {setting}")
        rows = stream_pair_metrics(song, setting)

        chunk_rows += [{"Song": song, "Setting": setting, **r} for r in rows]

        pair_rows.append(
            {
                "Split": split_name,
                "Song": song,
                "Setting": setting,
                "Duration (s)": sum(r["Frames"] for r in rows) / SAMPLE_RATE,
                **_weighted_average_metric_rows(rows, metric_cols),
            }
        )
        gc.collect()

    split_rows_all.append(
        {
            "Split": split_name,
            "Pairs": len(pairs_to_eval),
            "Chunks": len(chunk_rows),
            "Duration (s)": sum(r["Frames"] for r in chunk_rows) / SAMPLE_RATE,
            **_weighted_average_metric_rows(chunk_rows, metric_cols),
        }
    )

    pair_rows_all += pair_rows

pair_metrics_df = pd.DataFrame(pair_rows_all)
audio_metrics_df = pd.DataFrame(split_rows_all)

display(pair_metrics_df)
display(audio_metrics_df)


[validation 1/10] Ecstasy / threshold_-12_attack_10_release_0.4_ratio_10
[validation 2/10] Ecstasy / threshold_-12_attack_1_release_0.1_ratio_2
[validation 3/10] Ecstasy / threshold_-4_attack_10_release_0.1_ratio_2
[validation 4/10] Ecstasy / threshold_-4_attack_1_release_0.4_ratio_10
[validation 5/10] Ecstasy / threshold_-8_attack_30_release_0.8_ratio_4
[validation 6/10] Ecstasy / threshold_0_attack_3_release_0.8_ratio_4
[validation 7/10] Ecstasy / threshold_12_attack_3_release_0.8_ratio_2
[validation 8/10] Ecstasy / threshold_4_attack_10_release_0.1_ratio_10
[validation 9/10] Ecstasy / threshold_8_attack_1_release_0.1_ratio_10
[validation 10/10] Ecstasy / threshold_8_attack_30_release_0.4_ratio_2
[test 1/4] Air / threshold_-12_attack_10_release_0.4_ratio_10
[test 2/4] Air / threshold_-12_attack_1_release_0.1_ratio_2
[test 3/4] AncoraQui / threshold_-12_attack_10_release_0.4_ratio_10
[test 4/4] AncoraQui / threshold_-12_attack_1_release_0.1_ratio_2


,Split,Song,Setting,Duration (s),GR MAE (dB),MAE (L1),MSE (L2),ESR,MR-STFT,EDC,M_NRMSE,M_SF
0,validation,Ecstasy,threshold_-12_attack_10_release_0.4_ratio_10,257.741497,0.000399,0.000827,1.531106e-06,0.025348,0.039651,0.026723,0.037324,0.087457
1,validation,Ecstasy,threshold_-12_attack_1_release_0.1_ratio_2,257.741497,0.000394,0.000910,1.858106e-06,0.024008,0.036626,0.021073,0.025997,0.090391
2,validation,Ecstasy,threshold_-4_attack_10_release_0.1_ratio_2,257.741497,0.000189,0.001801,7.677547e-06,0.022212,0.017327,0.017123,0.014885,0.040686
3,validation,Ecstasy,threshold_-4_attack_1_release_0.4_ratio_10,257.741497,0.000400,0.001595,5.777509e-06,0.024628,0.040065,0.022661,0.035506,0.089660
4,validation,Ecstasy,threshold_-8_attack_30_release_0.8_ratio_4,257.741497,0.000192,0.001226,3.523967e-06,0.022424,0.017354,0.020728,0.018385,0.037322
5,validation,Ecstasy,threshold_0_attack_3_release_0.8_ratio_4,257.741497,0.000208,0.001806,7.666205e-06,0.022288,0.018634,0.019686,0.018580,0.041203
6,validation,Ecstasy,threshold_12_attack_3_release_0.8_ratio_2,257.741497,0.000148,0.002658,1.714812e-05,0.021310,0.008956,0.011584,0.005134,0.025240
7,validation,Ecstasy,threshold_4_attack_10_release_0.1_ratio_10,257.741497,0.000144,0.002777,1.874505e-05,0.021286,0.009165,0.012706,0.005734,0.026066
8,validation,Ecstasy,threshold_8_attack_1_release_0.1_ratio_10,257.741497,0.000145,0.002784,1.888298e-05,0.021050,0.008208,0.010860,0.002859,0.024536
9,validation,Ecstasy,threshold_8_attack_30_release_0.4_ratio_2,257.741497,0.000147,0.002583,1.617892e-05,0.021288,0.008921,0.012247,0.004915,0.025143


,Split,Pairs,Chunks,Duration (s),GR MAE (dB),MAE (L1),MSE (L2),ESR,MR-STFT,EDC,M_NRMSE,M_SF
0,validation,10,260,2577.414966,0.000237,0.001897,0.000010,0.022584,0.020491,0.017539,0.016932,0.04877
1,test,4,88,857.838005,0.000444,0.000836,0.000002,0.033012,0.045657,0.120837,0.048880,0.13834
